<a href="https://colab.research.google.com/github/hassan11196/llm-infra-lab/blob/main/notebooks/04_agents/01_react_from_scratch.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" style="height: 28px; vertical-align: middle;"/></a>  **[▶️ Run this notebook in Colab](https://colab.research.google.com/github/hassan11196/llm-infra-lab/blob/main/notebooks/04_agents/01_react_from_scratch.ipynb)**

# ReAct from scratch — an agent is just a loop

> **Track 04 — Agents · Notebook 01 · Runtime: ≈30 s on CPU**
>
> **Prerequisites:** you've written a Python REPL once. You know what a
> regex is.
>
> **What you'll know by the end:** what an "LLM agent" actually is under
> the hood (hint: there's no magic), how to build one without any
> framework, and how to score an agent's behaviour with numerical checks
> instead of vibes.

---

## An agent is not what the marketing says it is

If you've read "LLM agent" blog posts, you might think agents are some
new kind of software — reasoning machines, autonomous beings, etc. They
are not. An agent is a **three-line loop**:

```python
while not done:
    action = llm.propose(history)
    result = execute(action)
    history += f"\n{action}\nObservation: {result}"
```

That's it. The LLM proposes the next step, you run it (a tool call, a
web fetch, an arithmetic evaluation), you feed the result back into the
LLM's context, you loop. When the LLM says "I'm done, here's the
answer", you stop.

The 2022 paper that named this pattern *ReAct* (Yao et al.) proposed two
refinements:

1. Make the LLM emit a `Thought:` line before each action so its
   reasoning is legible (and parseable).
2. Constrain actions to a strict grammar, e.g. `Action: tool_name[argument]`,
   so the loop can dispatch reliably.

With those two moves you get something that looks and feels like an
agent — but mechanically it's the same loop as a REPL, just with an LLM
at the prompt instead of you.

### Why we're not calling a real LLM

Every serious agent framework (LangGraph, AutoGen, CrewAI) burns API
credits on every step. That's bad for a learning notebook — non-deterministic,
requires secrets, slow. So we build the same machinery around a
**deterministic rule-based policy** that emits ReAct-format strings
based on keyword matching. The parser, the tools, the state machine,
the scoring — all of it — is exactly what you'd have with a real LLM.
Only the "brain" is stubbed so the notebook runs anywhere.

A commented `VLLMPolicy` below shows exactly how to swap in a real
local model (Qwen2.5-0.5B-Instruct via vLLM or Ollama).


In [ ]:
from __future__ import annotations

import ast
import operator as op
import re
import sys
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

from llm_infra_lab._utils import set_seed
from scoring.harness import Scorer

set_seed(0)
s = Scorer("04_agents_01_react_from_scratch")


## Step 1 — The parser

Whatever the LLM generates, we need to turn it back into a structured
object with fields like `thought`, `action`, `action_input`, or
`final_answer`. Two regexes are enough:

- `Action:\s*(\w+)\[(.*?)\]` matches `Action: calculator[2 + 2]`.
- `Final Answer:\s*(.*)` matches the terminator.

If neither matches, the output is a **parse error** — the agent didn't
follow its own format. Parse errors are tracked as a separate metric
because a model that reasons brilliantly but can't stay in format is
useless in production.


In [ ]:
ACTION_RE = re.compile(r"Action:\s*(\w+)\[(.*?)\]", re.DOTALL)
FINAL_RE = re.compile(r"Final\s*Answer:\s*(.*)", re.DOTALL)


@dataclass
class ParsedStep:
    thought: str | None
    action: str | None
    action_input: str | None
    final_answer: str | None
    raw: str

    @property
    def is_action(self) -> bool: return self.action is not None

    @property
    def is_final(self) -> bool: return self.final_answer is not None


def parse_llm_output(text: str) -> ParsedStep:
    thought_match = re.search(r"Thought:\s*(.+?)(?=\n(?:Action|Final Answer):|$)", text, re.DOTALL)
    thought = thought_match.group(1).strip() if thought_match else None
    final_match = FINAL_RE.search(text)
    if final_match:
        return ParsedStep(thought, None, None, final_match.group(1).strip(), text)
    action_match = ACTION_RE.search(text)
    if action_match:
        return ParsedStep(thought, action_match.group(1), action_match.group(2).strip(), None, text)
    return ParsedStep(thought, None, None, None, text)


# Smoke test: does the parser find the action in a well-formed output?
example = "Thought: I need to compute 2+2.\nAction: calculator[2+2]"
parsed = parse_llm_output(example)
print(f"action = {parsed.action!r}  input = {parsed.action_input!r}")
assert parsed.action == "calculator"


## Step 2 — Tools are just Python functions

Every tool is a `str → str` function. The agent loop calls
`TOOLS[action](action_input)` and feeds the return value back as the
next `Observation:`.

Three tools:

- **`calculator`** — safely evaluates a simple arithmetic expression.
  Uses an AST walk instead of `eval` because running untrusted strings
  through `eval` is how you get remote code execution. A standard
  interview / security checkpoint.
- **`wiki_search`** — looks up a key in a dict of 25 stubbed facts. In
  a real system this would hit Wikipedia or a retrieval service; here
  it's a dict for determinism.
- **`get_datetime`** — returns the date relative to a *frozen* "now"
  so the notebook's output doesn't drift by day.

Notice: no abstract "Tool" class, no protocol, no registry pattern. Just
three functions and a dict. Frameworks can add more structure on top,
but the core is this simple.


In [ ]:
_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg, ast.UAdd: op.pos, ast.Mod: op.mod,
}


def _eval_ast(node: ast.AST) -> float:
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp):
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    if isinstance(node, ast.UnaryOp):
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.operand))
    raise ValueError(f"unsupported node {type(node).__name__}")


def calculator(expression: str) -> str:
    try:
        value = _eval_ast(ast.parse(expression.strip(), mode="eval").body)
    except Exception as e:  # noqa: BLE001 - tool outputs must never raise
        return f"ERROR: {type(e).__name__}: {e}"
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value) if not isinstance(value, float) else f"{value:.6f}".rstrip("0").rstrip(".")


FACTS = {
    "capital of france": "Paris", "capital of japan": "Tokyo", "capital of brazil": "Brasilia",
    "capital of canada": "Ottawa", "capital of kenya": "Nairobi", "capital of australia": "Canberra",
    "capital of south africa": "Pretoria", "capital of egypt": "Cairo",
    "capital of argentina": "Buenos Aires", "capital of mexico": "Mexico City",
    "tallest mountain": "Mount Everest at 8848 metres.",
    "longest river": "The Nile is 6650 km long.",
    "speed of light": "Approximately 299,792,458 metres per second in vacuum.",
    "electron charge": "Approximately 1.602e-19 coulombs.",
    "planck constant": "Approximately 6.626e-34 joule-seconds.",
    "number of continents": "Seven.",
    "chemical symbol for gold": "Au.", "chemical symbol for iron": "Fe.",
    "chemical symbol for silver": "Ag.", "chemical symbol for tungsten": "W.",
    "author of hamlet": "William Shakespeare.", "author of the iliad": "Homer.",
    "author of war and peace": "Leo Tolstoy.", "author of the great gatsby": "F. Scott Fitzgerald.",
    "painter of the mona lisa": "Leonardo da Vinci.",
}


def wiki_search(query: str) -> str:
    q = query.strip().lower().rstrip("?").strip()
    if q in FACTS:
        return FACTS[q]
    for key, val in FACTS.items():
        if key in q or q in key:
            return val
    return "No matching result in the local knowledge base."


FIXED_NOW = datetime(2026, 4, 17, 12, 0, tzinfo=timezone.utc)


def get_datetime(spec: str) -> str:
    spec = spec.strip().lower()
    if spec in {"", "now", "today", "current date"}: return FIXED_NOW.date().isoformat()
    if spec == "year":     return str(FIXED_NOW.year)
    if spec == "month":    return FIXED_NOW.strftime("%B")
    if spec == "day":      return FIXED_NOW.strftime("%A")
    if spec == "tomorrow": return (FIXED_NOW + timedelta(days=1)).date().isoformat()
    if spec == "yesterday":return (FIXED_NOW - timedelta(days=1)).date().isoformat()
    return f"ERROR: unknown date spec {spec!r}"


TOOLS: dict[str, callable] = {
    "calculator": calculator, "wiki_search": wiki_search, "get_datetime": get_datetime,
}
print("tools:", list(TOOLS))


## Step 3 — The "brain"

A policy is an object with one method: `step(task, scratch) -> str`.
Given the user's task and the running transcript, return the next
ReAct-format string.

We ship two policies:

- **`RuleBasedPolicy`** — keyword matches the task to a tool. Fully
  deterministic. Used for scoring.
- **`VLLMPolicy`** — commented sketch for swapping in a local LLM.
  Points at an OpenAI-compatible endpoint (vLLM or Ollama) running
  `Qwen/Qwen2.5-0.5B-Instruct`.

The punchline: **the policy is the only part of ReAct that needs an
LLM.** The parser, the tools, the loop, and the scoring work
identically whether the brain is GPT-5 or a Python `if` statement. The
framework sellers don't usually put it this bluntly.


In [ ]:
class Policy:
    def step(self, prompt: str, scratch: str) -> str:
        raise NotImplementedError


class RuleBasedPolicy(Policy):
    '''Deterministic stand-in for an LLM. Routes each question to the right tool.'''

    def step(self, prompt: str, scratch: str) -> str:
        # Once we have an Observation from a tool, finalise.
        if "Observation:" in scratch:
            last_obs = scratch.rsplit("Observation:", 1)[1].strip()
            return f"Thought: I have the answer now.\nFinal Answer: {last_obs}"

        q = prompt.lower()

        if any(w in q for w in ("compute", "evaluate", "what is", "calculate", "sum", "product")):
            expr = re.sub(r"[^0-9+\-*/%().^ \t]", " ", prompt).strip()
            if expr:
                return f"Thought: I need a calculator.\nAction: calculator[{expr}]"

        if any(w in q for w in ("today", "current", "date", "what year", "what month",
                                 "day of the week", "tomorrow", "yesterday")):
            arg = ("year" if "year" in q else "month" if "month" in q
                   else "day" if "day of" in q else "tomorrow" if "tomorrow" in q
                   else "yesterday" if "yesterday" in q else "today")
            return f"Thought: I need the current date.\nAction: get_datetime[{arg}]"

        return f"Thought: I should look this up.\nAction: wiki_search[{prompt.rstrip('?').strip()}]"


class VLLMPolicy(Policy):
    '''Swap-in for a local vLLM / Ollama endpoint serving Qwen2.5-0.5B-Instruct.

    Set ``VLLM_URL`` to e.g. http://localhost:8000/v1/chat/completions and
    ``VLLM_MODEL`` to the served model name.
    '''

    def __init__(self, url: str, model: str) -> None:
        self.url = url; self.model = model

    def step(self, prompt: str, scratch: str) -> str:
        import json, urllib.request
        req = urllib.request.Request(
            self.url,
            data=json.dumps({
                "model": self.model,
                "messages": [
                    {"role": "system", "content": (
                        "You are a ReAct agent. Respond with 'Thought:' then "
                        "'Action: calculator[expr]', 'Action: wiki_search[q]', "
                        "'Action: get_datetime[today|year|month|day|tomorrow|yesterday]', "
                        "or 'Final Answer: <answer>'."
                    )},
                    {"role": "user", "content": prompt + "\n" + scratch},
                ],
                "temperature": 0,
            }).encode(),
            headers={"Content-Type": "application/json"},
        )
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())["choices"][0]["message"]["content"]


## Step 4 — The ReAct loop

Six lines of actual logic: call policy, parse output, dispatch to tool,
append observation, stop if final answer or max steps. Everything else
is bookkeeping.


In [ ]:
@dataclass
class EpisodeResult:
    task: str
    expected: str
    final: str | None
    steps: int
    parse_errors: int
    transcript: str


def run_episode(task: str, expected: str, policy: Policy, max_steps: int = 6) -> EpisodeResult:
    scratch = ""
    parse_errors = 0
    for step in range(max_steps):
        out = policy.step(task, scratch)
        parsed = parse_llm_output(out)

        if parsed.is_final:
            return EpisodeResult(task, expected, parsed.final_answer, step + 1, parse_errors, scratch + out)

        if not parsed.is_action:
            parse_errors += 1
            scratch += f"\n{out}\nObservation: parse error; expected an Action or Final Answer.\n"
            continue

        tool = TOOLS.get(parsed.action)
        if tool is None:
            parse_errors += 1
            scratch += f"\n{out}\nObservation: unknown tool {parsed.action!r}\n"
            continue

        obs = tool(parsed.action_input or "")
        scratch += f"\n{out}\nObservation: {obs}\n"

    return EpisodeResult(task, expected, None, max_steps, parse_errors, scratch)


def grade(result: EpisodeResult) -> bool:
    if result.final is None: return False
    return result.expected.lower() in result.final.lower()


## Step 5 — 20 tasks across three kinds of queries

Eight arithmetic, eight lookup, four temporal. Ground truth for each is
a single substring that should appear in the final answer.

Three metrics:

- **Success rate** — fraction of tasks solved correctly. Floor: 70 %.
- **Parse-error rate** — fraction of LLM outputs that didn't follow the
  format. Ceiling: 10 %.
- **Avg steps per task** — how many loop iterations it takes on
  average. Ceiling: 5. (If your agent is bouncing around in 10+ steps
  per task, it's wandering.)


In [ ]:
TASKS: list[tuple[str, str]] = [
    # arithmetic
    ("Compute 12 + 30", "42"), ("What is 7 * 8?", "56"),
    ("Calculate (15 + 5) / 4", "5"), ("Compute 2 ** 10", "1024"),
    ("Evaluate 100 - 37", "63"), ("What is 1234 + 5678?", "6912"),
    ("Compute 9 * 9 - 1", "80"), ("Calculate 144 / 12", "12"),
    # lookup
    ("What is the capital of France?", "Paris"),
    ("What is the capital of Japan?", "Tokyo"),
    ("What is the tallest mountain?", "Everest"),
    ("Who wrote Hamlet?", "Shakespeare"),
    ("What is the chemical symbol for gold?", "Au"),
    ("What is the speed of light?", "299,792,458"),
    ("Who painted the Mona Lisa?", "Leonardo"),
    ("How many continents are there?", "Seven"),
    # temporal
    ("What is the current year?", "2026"),
    ("What is the current month?", "April"),
    ("What is the date tomorrow?", "2026-04-18"),
    ("What is the date yesterday?", "2026-04-16"),
]

policy = RuleBasedPolicy()
results = [run_episode(task, expected, policy) for task, expected in TASKS]

n_success = sum(grade(r) for r in results)
total_parse_errors = sum(r.parse_errors for r in results)
avg_steps = sum(r.steps for r in results) / len(results)
success_rate = n_success / len(results)
parse_err_rate = total_parse_errors / sum(r.steps for r in results)

for r in results[:5]:
    print(f"{r.task:<45}  expected={r.expected!r:<12}  final={r.final!r}")
print(f"\nsuccess {n_success}/{len(results)} ({success_rate:.0%})  "
      f"avg steps {avg_steps:.2f}  parse errors {parse_err_rate:.1%}")


In [ ]:
s.check("success_rate_at_least_70pct", lambda: success_rate >= 0.70, msg=f"{success_rate:.2%}")
s.check("parse_error_rate_at_most_10pct", lambda: parse_err_rate <= 0.10, msg=f"{parse_err_rate:.2%}")
s.check("avg_steps_at_most_5", lambda: avg_steps <= 5.0, msg=f"{avg_steps:.2f}")
s.check("every_task_has_finite_trace", lambda: all(r.steps >= 1 for r in results))
s.check(
    "parser_extracts_action_from_well_formed_output",
    lambda: parse_llm_output("Thought: x\nAction: calculator[2+2]").action == "calculator",
)


## Exercises

1. **Swap the brain for a real model.** Start a local vLLM server (or
   `ollama run qwen2.5:0.5b`), point `VLLMPolicy` at it, rerun the
   tasks. Watch parse-error rate go up — a tiny instruct model
   occasionally misses the format — and invent a retry strategy.
2. **Add a new tool.** Write a `search_pypi(package)` function that
   returns a short description of a Python package. Plug it into
   `TOOLS`, add a keyword route in `RuleBasedPolicy`, and add a few
   tasks to test it.
3. **Force format with Outlines.** The next notebook in this track
   (`02_structured_outputs_three_ways`) uses grammar-constrained
   decoding to guarantee parseable output. After reading it, come back
   here and replace the parser's "parse error" path with a
   grammar-constrained regeneration.
4. **Multi-hop queries.** Add a task like "What is the population of
   the capital of Japan?" where the agent needs two tool calls.
   Current policy fails — think about what state the policy needs to
   track to chain correctly.

## Further reading

- Yao et al. 2022, *ReAct: Synergizing Reasoning and Acting in Language
  Models* (arxiv 2210.03629).
- Schick et al. 2023, *Toolformer* — how tool-use can be learned
  instead of prompted.
- Anthropic 2024 blog, *Building Effective Agents* — empirical notes
  on when simple loops beat framework-heavy designs.

## What's next

Notebook 02 (`02_structured_outputs_three_ways`) fixes the parse-error
problem with grammar-constrained decoding. Notebook 03
(`03_langgraph_state_machines`) takes the same ideas to a proper state
machine with branches and retries. By notebook 05 we're talking to
real tools via the MCP protocol.


In [ ]:
s.summary()
s.save()
